# A Modular Planner/Executor/Validator Agent (arxiv:2310.00194)

"Improving Planning with Large Language Models: A Modular Agentic
Architecture" (https://arxiv.org/pdf/2310.00194) separates planning
cognition from execution instead of one end-to-end generation: a
**Planner** decomposes the task into steps, an **Executor** carries the
plan out, and a **Validator** checks the result — sending the loop back to
the planner with feedback (informed by a rolling **Memory** of prior
attempts) when it isn't good enough, up to a bounded number of passes.

Nothing like this exists in psychscanner today (the built-in agent is one
LLM call per trial), so `psychscanner.agents.make_planner_executor_agent`
builds the loop directly on LangGraph and adapts it to the `ScanningAgent`
contract. Uses the local `llama3.2:3b` Ollama model — no tool calling
needed here, just plain chat completions.

In [1]:
from pathlib import Path

from langchain_core.messages import HumanMessage

import psychscanner as psy
from psychscanner.agents import make_planner_executor_agent
from psychscanner.memories import llm_chat_model
from psychscanner.task_runner import TaskRunner

model = llm_chat_model(model="llama3.2:3b", family="ollama", parameters={"temperature": 0})
agent = make_planner_executor_agent(model, max_iterations=2)
print("PsychScanner successfully imported!")

--<api key>-- warning: OLLAMA_API_KEY not set; proceeding without explicit api_key for family 'ollama'


--<chat model>-- metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}} output_version=None model='llama3.2:3b' temperature=0.0


PsychScanner successfully imported!


## 1. Directly through `TaskRunner`

In [2]:
tasktrials = {"trials": [
    {"trcode": "t1", "stimulus": HumanMessage(content="A survey respondent rates their mood 2/5 and energy 4/5. "
                                                        "In one sentence, what does this combination suggest?"),
     "tasktype": "x", "parser": None, "fb": False},
]}
runner = TaskRunner(
    scanning_agent=agent, trace_cfg={"trial": "tut-", "task": "tut-task"},
    system_message="You are a careful research assistant.",
    tasktrials=tasktrials, chain_type="item", hmsg="stimulus",
)
for r in runner.execute():
    print(r["trcode"], "->", r["pred_resp"].content)

----<>---- task running


t1 -> I will follow the plan to provide the final answer:

1. Analyze the given ratings: The respondent's mood is rated as 2/5 (low), and their energy is rated as 4/5 (high).
2. Consider the possible reasons for these ratings: A low mood rating could be due to various factors such as stress, illness, or lack of motivation.
3. Combine the ratings to draw a conclusion: The high energy rating suggests that despite feeling low, the respondent may still have some motivation or enthusiasm left.
4. Formulate a sentence summarizing the combination: This combination suggests that the respondent is experiencing a state of low motivation and energy, but still has some residual drive or enthusiasm.

Final Answer: The respondent's low mood rating combined with their high energy rating suggests they are feeling unmotivated but still have some spark left.


## 2. Through `ScannerModel.run(custom_agent=...)`

Same seam as any `ScanningAgent`: `custom_agent` bypasses `AgentConfig`'s
LangChain model, so the card's `model`/`family` don't matter. `nsim=2`
simulates two independent participants through the same planner/executor/
validator loop.

In [3]:
RUN_DIR = Path.cwd() / "_planner_executor_tutorial_run"

task = {
    "tasktype": "survey", "taskname": "planner_executor_demo",
    "instructions": {"definition": ["Answer thoughtfully, showing your reasoning is sound."]},
    "contexts": ["General reasoning"], "contexts_id": ["gr"], "context_present": False,
    "chain_type": "item", "parser": "0",
    "items": {"gr": [
        {"trcode": "gr_1", "stimulus": "If a study has 40 participants and 8 drop out, what fraction completed it?"},
    ]},
}

card_in = psy.ExpCardInit()
card_in.proj_dir, card_in.projectname = RUN_DIR, "planner_executor_demo"
card_in.task_file = task
card_in.cogtype, card_in.nsim = "no", 2
card_in.chain_type, card_in.memory = "item", "SingleTurn"

expcard = psy.ExpCard(card_in)
scanner = psy.ScannerModel(expcard=expcard)
scan_results = scanner.run(custom_agent=agent)
for participant in scan_results:
    for trial in participant:
        print(trial["trcode"], "->", trial["pred_resp"].content)

----<PROJECT AND DATA ROOT DIRECTORY>----


	Project root dir: /Users/saurabhext/Documents/PSYCHSCANNER/psychscanner/examples/_planner_executor_tutorial_run


	Simulation data root dir: /Users/saurabhext/Documents/PSYCHSCANNER/psychscanner/examples/_planner_executor_tutorial_run/planner_executor_demo/planner_executor_demo/mock-llm_mock-chat-model_SingleTurn


----<>----


--<chat model>-- metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}} output_version=None model_name='mock-chat-model' repeat_buffer_length=10


...

1it [00:57, 57.92s/it]


2026-07-06 11:54:09.661 | INFO     | psychscanner.session_tunnel.session_tunnel:scan_checkpoint:184 - scan-checkpoint


----<scanned runs>---- i = 1


2026-07-06 11:54:09.667 | CRITICAL | psychscanner.session_tunnel.session_tunnel:end_checkpoint:163 - END


gr_1 -> I will carry out the plan as specified.

**Step 1: Identify the total number of participants in the study.**
The total number of participants is given as 40.

**Step 2: Determine the number of participants who dropped out.**
The number of participants who dropped out is given as 8.

**Step 3: Subtract the number of participants who dropped out from the total number of participants to find the number of participants who completed the study.**
Number of participants who completed the study = Total number of participants - Number of participants who dropped out
= 40 - 8
= 32

**Step 4: Write the fraction with the number of participants who completed the study as the numerator and the total number of participants as the denominator.**
Fraction = (Number of participants who completed the study) / (Total number of participants)
= 32/40

To simplify this fraction, we can divide both the numerator and the denominator by their greatest common divisor, which is 8.

Simplified Fraction = 

## Recap

- Planner, Executor, and Validator are separate LangGraph nodes, not one
  end-to-end generation — the Validator can loop the trajectory back to the
  Planner (bounded by `max_iterations`) instead of accepting the first draft.
- A rolling `memory` list threads prior plans/executions/feedback into each
  new planning pass.
- Plugs into psychscanner exactly like any other `ScanningAgent` —
  `TaskRunner(scanning_agent=...)` directly, or
  `ScannerModel.run(custom_agent=...)` for a full simulation.

---
## Further reading

Advanced applications of the planner/executor/validator architecture:

1. **["Improving Planning with Large Language Models: A Modular Agentic Architecture"](https://arxiv.org/abs/2310.00194)** — the source paper for `make_planner_executor_agent` itself, with the full ablation of how the bounded validator-feedback loop improves planning over one-shot generation.
2. **["Plan-and-Act: Improving Planning of Agents for Long-Horizon Tasks"](https://arxiv.org/abs/2503.09572)** (2025) — separates planning from execution at a larger scale (long-horizon web tasks), the same decomposition principle behind the Planner/Executor split here.